In [37]:
!pip install umap-learn

In [3]:
import os
os.chdir(r"E:\Text analysis")

print("Jupyter has successfully moved to：", os.getcwd())

✅ Jupyter 已经成功搬家到了： E:\Text analysis


In [5]:
import pandas as pd

files = {
    'edu': '2011qualification.csv',
    'occ': '2011occupations.csv',
    'health': '2011health.csv'
}

# clean the data
def clean_ons_flexible(file_path):
    temp_df = pd.read_csv(file_path, nrows=20, header=None)
    header_row = 0
    for i, row in temp_df.iterrows():
        if 'Area code' in str(row.values) or 'geography code' in str(row.values):
            header_row = i
            break
            
    df = pd.read_csv(file_path, header=header_row)

    df = df.rename(columns={'geography code': 'Area code', 'mnemonic': 'Area code'})

    data_start_col = [col for col in df.columns if 'All categories' in col or 'Total' in col][0]
    data_idx = list(df.columns).index(data_start_col)

    if 'geography' in df.columns:
        df = df.rename(columns={'geography': 'Area name'})
    else:
        name_cols = list(df.columns[1:data_idx])
        df['Area name'] = df[name_cols].bfill(axis=1).iloc[:, 0]
    
    keep_cols = ['Area code', 'Area name'] + list(df.columns[data_idx:])
    df = df[keep_cols]
    df = df.dropna(axis=1, how='all').dropna(subset=['Area code'])
    df = df[df['Area code'].str.match(r'^[A-Z][0-9]{8}$', na=False)]
    df.columns = [col.replace('\n', ' ').strip() for col in df.columns]
    
    return df

print("merging the data")

# loading the data
df_edu = clean_ons_flexible(files['edu'])
df_occ = clean_ons_flexible(files['occ'])
df_health = clean_ons_flexible(files['health'])

df_occ_clean = df_occ.drop(columns=['Area name'])
df_health_clean = df_health.drop(columns=['Area name'])

# combine the data
merged_1 = pd.merge(df_edu, df_occ_clean, on='Area code', how='inner')
final_df = pd.merge(merged_1, df_health_clean, on='Area code', how='inner')

print(f"Make it！Remain {final_df.shape[0]} cities.")
display(final_df)
# save as CSV file
file_name = 'Census_2011_Cleaned_Original.csv'
final_df.to_csv(file_name, index=False, encoding='utf-8-sig')

print(f"The final file is saved as：{file_name}")

merging the data
Make it！Remain 342 cities.


,Area code,Area name,All categories: Highest level of qualification,No qualifications,Highest level of qualification: Level 1 qualifications,Highest level of qualification: Level 2 qualifications,Highest level of qualification: Apprenticeship,Highest level of qualification: Level 3 qualifications,Highest level of qualification: Level 4 qualifications and above,Highest level of qualification: Other qualifications,...,L14.2 Long-term unemployed,Not classified,L15 Full-time students,L17 Not classifiable for other reasons,General Health: All categories: General health; measures: Value,General Health: Very good health; measures: Value,General Health: Good health; measures: Value,General Health: Fair health; measures: Value,General Health: Bad health; measures: Value,General Health: Very bad health; measures: Value
0,E06000047,County Durham UA,"425,258",27.5,13.4,16.0,4.2,13.6,21.5,3.9,...,2.0,2.4,2.4,0.0,513242,217373,171564,82404,32568,9333
1,E06000005,Darlington UA,"85,357",24.8,13.4,15.9,5.1,12.7,23.7,4.3,...,2.4,1.1,1.1,0.0,105564,47046,37199,15116,4763,1440
2,E06000001,Hartlepool UA,"74,228",30.7,13.5,16.2,5.5,12.5,17.6,4.0,...,3.8,1.1,1.1,0.0,92028,39816,30117,14607,5789,1699
3,E06000002,Middlesbrough UA,"110,409",29.9,13.6,15.3,4.4,13.2,18.5,5.0,...,3.6,4.1,4.1,0.0,138412,62855,45295,19719,8001,2542
4,E06000003,Redcar and Cleveland UA,"111,011",28.4,13.5,16.4,5.7,13.1,18.9,3.9,...,3.3,1.0,1.0,0.0,135177,57285,45844,21449,8184,2415
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
337,W06000018,Caerphilly,"143,825",31.4,14.9,16.2,3.9,11.2,18.7,3.7,...,2.5,0.9,0.9,0.0,178806,79382,54272,28488,12844,3820
338,W06000019,Blaenau Gwent,"57,321",36.0,15.8,15.5,3.5,9.8,15.2,4.2,...,3.8,0.9,0.9,0.0,69814,29269,21385,11696,5692,1772
339,W06000020,Torfaen,"73,833",28.9,15.0,16.3,4.3,11.2,20.3,4.0,...,2.1,0.7,0.7,0.0,91075,39398,29619,14155,5965,1938
340,W06000021,Monmouthshire,"75,080",20.7,12.6,15.4,3.9,10.8,33.1,3.6,...,1.1,0.7,0.7,0.0,91323,42365,30742,12800,4173,1243


The final file is saved as：Census_2011_Cleaned_Original.csv


In [16]:
import pandas as pd

# define the cleaning function
# ==========================================
def clean_ons_2021(file_path):
    df = pd.read_csv(file_path)
    
    rename_dict = {}
    for col in df.columns:
        col_lower = str(col).lower()
        if 'code' in col_lower and ('geography' in col_lower or 'area' in col_lower or 'local authority' in col_lower):
            rename_dict[col] = 'Area code'
        elif 'geography' in col_lower or 'area name' in col_lower or 'local authority' in col_lower:
            if 'code' not in col_lower: 
                rename_dict[col] = 'Area name'
                
    df = df.rename(columns=rename_dict)
    
    if 'Area code' in df.columns:
        df = df.dropna(subset=['Area code'])
        df = df[df['Area code'].str.match(r'^[EW][0][6-9][0-9]{6}$', na=False)]
    else:
        print(f"Warning：Cannot find the area code in {file_path} ！")
        return df
        
    df.columns = [str(col).replace('\n', ' ').strip() for col in df.columns]
    
    return df

file_edu_2021 = '2021qualification.csv'
file_occ_2021 = '2021occupations.csv'
file_health_2021 = '2021health.csv' 

print("cleaning the data of 2021...")
df_edu_2021 = clean_ons_2021(file_edu_2021)
df_occ_2021 = clean_ons_2021(file_occ_2021)
df_health_2021 = clean_ons_2021(file_health_2021) 

print("merging several tables")
merged_1 = pd.merge(df_edu_2021, df_occ_2021, on=['Area code', 'Area name'], how='inner')
final_df_2021 = pd.merge(merged_1, df_health_2021, on=['Area code', 'Area name'], how='inner')

print(f"Make it！The data of 2021 has been produced！Contains {len(final_df_2021)} cities。")

# check the variables and save the file
print("\n--- 2021版 合并后的全变量目录 (包含Edu, Occ, Health) ---")
for i, col in enumerate(final_df_2021.columns):
    print(f"{i}. {col}")

final_df_2021.to_csv('Census_2021_Cleaned_Original.csv', index=False, encoding='utf-8-sig')
print("\nThe file is saved as: Census_2021_Cleaned_Original.csv")

🚀 正在清洗 2021 年铁三角数据...
🔗 正在进行多表缝合...
✅ 缝合大功告成！2021 年铁三角表格已生成！包含 331 个城市。

--- 2021版 合并后的全变量目录 (包含Edu, Occ, Health) ---
0. date_x
1. Area name
2. Area code
3. Highest level of qualification: Total: All usual residents aged 16 years and over
4. Highest level of qualification: No qualifications
5. Highest level of qualification: Level 1 and entry level qualifications
6. Highest level of qualification: Level 2 qualifications
7. Highest level of qualification: Apprenticeship
8. Highest level of qualification: Level 3 qualifications
9. Highest level of qualification: Level 4 qualifications and above
10. Highest level of qualification: Other qualifications
11. date_y
12. National Statistics Socio-economic Classification (NS-SEC): Total: All usual residents aged 16 years and over
13. National Statistics Socio-economic Classification (NS-SEC): L1, L2 and L3 Higher managerial, administrative and professional occupations
14. National Statistics Socio-economic Classification (NS-SEC): L4, L5 and L6

In [17]:
# ==========================================
# 找出 2011 年存在，但 2021 年消失的地区
# ==========================================
# 提取两年各自的地区名字集合
areas_2011 = set(final_df['Area name'])
areas_2021 = set(final_df_2021['Area name'])

# 集合相减：2011年有，但2021年没有的
missing_in_2021 = areas_2011 - areas_2021

print(f"🔍 调查结果：共有 {len(missing_in_2021)} 个地区在 2011 年存在，但在 2021 年消失（通常是因为合并或改名）。\n")
print("--- 消失的地区名单 ---")

# 把名单按字母排序并打印出来
for i, area in enumerate(sorted(missing_in_2021)):
    print(f"{i+1}. {area}")

# (可选) 如果你还想看看有没有2021年凭空“新增”的地区：
new_in_2021 = areas_2021 - areas_2011
print(f"\n✨ 顺便发现：共有 {len(new_in_2021)} 个全新命名的地区在 2021 年诞生！")

🔍 调查结果：共有 89 个地区在 2011 年存在，但在 2021 年消失（通常是因为合并或改名）。

--- 消失的地区名单 ---
1. Aylesbury Vale
2. Bath and North East Somerset UA
3. Bedford UA
4. Blackburn with Darwen UA
5. Blackpool UA
6. Bournemouth UA
7. Bracknell Forest UA
8. Bridgend 
9. Brighton and Hove UA
10. Bristol, City of UA
11. Central Bedfordshire UA
12. Cheshire East UA
13. Cheshire West and Chester UA
14. Chiltern
15. Christchurch
16. Corby
17. Cornwall UA
18. County Durham UA
19. Darlington UA
20. Daventry
21. Derby UA
22. Dudley 
23. East Dorset
24. East Northamptonshire
25. East Riding of Yorkshire UA
26. Forest Heath
27. Halton UA
28. Hartlepool UA
29. Herefordshire, County of UA
30. Isle of Wight UA
31. Isles of Scilly UA
32. Kettering
33. Kingston upon Hull, City of UA
34. King’s Lynn and West Norfolk
35. Kirklees 
36. Knowsley 
37. Leicester UA
38. Luton UA
39. Medway UA
40. Middlesbrough UA
41. Milton Keynes UA
42. North Dorset
43. North East Lincolnshire UA
44. North Lincolnshire UA
45. North Somerset UA
46. Northamp

In [23]:
import pandas as pd
import numpy as np

file_2011 = 'Census_2011_Cleaned_Original.csv' 
file_2021 = 'Census_2021_Cleaned_Original.csv'
file_lookup = 'Local_Authority_District_(2011)_to_Local_Authority_District_(2021)_Lookup_for_England_and_Wales.csv'

# 1. 读取数据
df_2011 = pd.read_csv(file_2011)
df_2021 = pd.read_csv(file_2021)
df_lookup = pd.read_csv(file_lookup)

print("🔄 正在执行高精度的地区对齐与数据聚合...")

# ==========================================
# 2. ⚡️修复一：使用“身份证号(Code)”进行精准映射⚡️
# ==========================================
# 构建 2011 Code -> 2021 Code 的映射
code_mapping = dict(zip(df_lookup['LAD11CD'], df_lookup['LAD21CD']))
# 构建 2021 Code -> 2021 Name 的映射
name_mapping = dict(zip(df_lookup['LAD21CD'], df_lookup['LAD21NM']))

df_2011_upgraded = df_2011.copy()

# 映射出 2021 年的新边界代码和新名字
df_2011_upgraded['Area code 2021'] = df_2011_upgraded['Area code'].map(code_mapping).fillna(df_2011_upgraded['Area code'])
df_2011_upgraded['Area name 2021'] = df_2011_upgraded['Area code 2021'].map(name_mapping).fillna(df_2011_upgraded['Area name'])


# ==========================================
# 3. ⚡️修复二：清除逗号陷阱，并自动包含 Health 变量⚡️
# ==========================================
# 智能识别出所有需要聚合的“数据列”（跳过名字和代码列）
numeric_cols = [col for col in df_2011_upgraded.columns if col not in ['Area code', 'Area name', 'Area code 2021', 'Area name 2021']]

# 批量清除数字里的逗号，并强制转换为浮点数，防止聚合时出错！
for col in numeric_cols:
    if df_2011_upgraded[col].dtype == object: # 如果被 Pandas 当成了字符串
        df_2011_upgraded[col] = df_2011_upgraded[col].astype(str).str.replace(',', '', regex=True).astype(float)

# 执行聚合：按新边界【代码】分组，将 2011 年的小城市数据加总合并成 2021 的大城市
df_2011_aligned = df_2011_upgraded.groupby(['Area code 2021', 'Area name 2021'])[numeric_cols].sum().reset_index()

# 把表头改回标准样式
df_2011_aligned.rename(columns={'Area code 2021': 'Area code', 'Area name 2021': 'Area name'}, inplace=True)


# ==========================================
# 4. ⚡️修复三：2021 年数据不需要二次清洗⚡️
# ==========================================
# 因为你刚刚在上一步已经用 clean_ons_2021 洗得干干净净了，直接用就行
df_2021_cleaned = df_2021.copy()


# ==========================================
# 5. 导出为精美 Excel (多 Sheet)
# ==========================================
output_name = 'UK_Census_Comparison_2011_2021.xlsx'
print(f"💾 正在生成终极 Excel 对比表: {output_name}")

with pd.ExcelWriter(output_name, engine='openpyxl') as writer:
    # Sheet 1: 完美对齐 2021 边界的 2011 铁三角数据
    df_2011_aligned.to_excel(writer, sheet_name='2011_Data_Aligned', index=False)
    
    # Sheet 2: 2021 铁三角数据
    df_2021_cleaned.to_excel(writer, sheet_name='2021_Cleaned', index=False)
    
    # Sheet 3: 原始地区对照表 (留作论文附件备用)
    df_lookup.to_excel(writer, sheet_name='District_Lookup_Map', index=False)

print(f"✅ 大功告成！2011 年聚合后剩余 {len(df_2011_aligned)} 个城市，准备好与 2021 年无缝对接！")

🔄 正在执行高精度的地区对齐与数据聚合...
💾 正在生成终极 Excel 对比表: UK_Census_Comparison_2011_2021.xlsx
✅ 大功告成！2011 年聚合后剩余 325 个城市，准备好与 2021 年无缝对接！


In [25]:
# ==========================================
# 核查 2011转换后 vs 2021原始表 的地区匹配度
# ==========================================

# 提取两份表格里的“地区名称”集合 (注意：请确保变量名和你运行的一致)
# 如果你的 2021 年表叫别的名字，比如 df_edu_2021，请自行替换下方变量名
areas_2011_aligned = set(df_2011_aligned['Area name'])
areas_2021 = set(final_df_2021['Area name']) 

# 1. 计算完全匹配的地区
common_areas = areas_2011_aligned.intersection(areas_2021)
print(f"✅ 完美匹配的地区数量：{len(common_areas)} 个")

# 2. 检查 2011 年对齐后，依然“多出来”的地区 (没能在 2021 找到)
only_in_2011 = areas_2011_aligned - areas_2021
print(f"\n⚠️ 仅在 2011 转换表中存在的地区数量：{len(only_in_2011)} 个")
if len(only_in_2011) > 0:
    print("它们是：")
    for area in sorted(list(only_in_2011)):
        print(f" - {area}")

# 3. 检查 2021 年表里“多出来”的地区 (没能在 2011 找到)
only_in_2021 = areas_2021 - areas_2011_aligned
print(f"\n⚠️ 仅在 2021 原始表中存在的地区数量：{len(only_in_2021)} 个")
if len(only_in_2021) > 0:
    print("它们是：")
    for area in sorted(list(only_in_2021)):
        print(f" - {area}")

# 4. 给出最终建议
if len(only_in_2011) == 0 and len(only_in_2021) == 0:
    print("\n🎉 结论：100% 完美对齐！两张表的地区完全一致，可以放心合并 (Merge) 了！")
else:
    print("\n🔍 结论：存在少量无法对齐的地区，合并时建议使用 how='inner' 取交集，保证数据不出错。")

✅ 完美匹配的地区数量：324 个

⚠️ 仅在 2011 转换表中存在的地区数量：1 个
它们是：
 - Rhondda Cynon Taf

⚠️ 仅在 2021 原始表中存在的地区数量：7 个
它们是：
 - East Hertfordshire
 - Gateshead
 - Northumberland
 - Rhondda Cynon Taff
 - St Albans
 - Stevenage
 - Welwyn Hatfield

🔍 结论：存在少量无法对齐的地区，合并时建议使用 how='inner' 取交集，保证数据不出错。


In [42]:
import pandas as pd

print("🤝 正在合并 2011 和 2021 的最终数据...")

# 💡 优化1：改用 'Area code' 合并！完美破解 "Taf" 和 "Taff" 的拼写差异！
final_merged = pd.merge(df_2011_aligned, df_2021_cleaned, on='Area code', how='inner')

# 处理合并后可能多出来的名字列
if 'Area name_x' in final_merged.columns:
    final_merged = final_merged.rename(columns={'Area name_x': 'Area name'}).drop(columns=['Area name_y'])

display(final_merged)
print(f"🎉 终极对齐成功！完美匹配了 {len(final_merged)} 个平级城市！")

final_merged.to_csv('ULTIMATE_2011_2021_Merged.csv', index=False, encoding='utf-8-sig')
print("💾 完美合并表已保存为: ULTIMATE_2011_2021_Merged.csv\n")
print("-" * 50)


# ==========================================
# 0. 读取总表，并执行“防爆破”数字清洗
# ==========================================
df_mixed = pd.read_csv('ULTIMATE_2011_2021_Merged.csv')

print("🧹 正在清理数字格式 (洗除隐藏的逗号，防止计算报错)...")
import pandas as pd
import numpy as np

print("🚀 启动【绝对人数+百分比】双轨制终极引擎...")

# ==========================================
# 1. 加载文件与基础数字清理
# ==========================================
df_2011 = pd.read_csv('Census_2011_Cleaned_Original.csv')
df_2021 = pd.read_csv('Census_2021_Cleaned_Original.csv')
df_lookup = pd.read_csv('Local_Authority_District_(2011)_to_Local_Authority_District_(2021)_Lookup_for_England_and_Wales.csv')

# 清洗数字逗号陷阱
for df_temp in [df_2011, df_2021]:
    for col in df_temp.columns:
        if col not in ['Area code', 'Area name', 'date']:
            if df_temp[col].dtype == object:
                df_temp[col] = pd.to_numeric(df_temp[col].astype(str).str.replace(',', '', regex=True), errors='coerce')

# ==========================================
# 2. 还原 2011 绝对人数 (解决合并统计的致命Bug)
# ==========================================
col_2011_edu_total = 'All categories: Highest level of qualification'
col_2011_occ_total = 'All categories: NS-SeC'

for col in df_2011.columns:
    if col in ['Area code', 'Area name', col_2011_edu_total, col_2011_occ_total]:
        continue
    
    if df_2011[col].max() <= 150: 
        if 'General Health' in col:
            continue
        elif 'L' in col or 'occupations' in col.lower() or 'unemployed' in col.lower() or 'student' in col.lower() or 'Not classified' in col:
            df_2011[col] = (df_2011[col] / 100.0) * df_2011[col_2011_occ_total]
        else:
            df_2011[col] = (df_2011[col] / 100.0) * df_2011[col_2011_edu_total]

# ==========================================
# 3. 时空对齐 (安全地累加绝对人数)
# ==========================================
code_mapping = dict(zip(df_lookup['LAD11CD'], df_lookup['LAD21CD']))
name_mapping = dict(zip(df_lookup['LAD21CD'], df_lookup['LAD21NM']))

df_2011['Area code 2021'] = df_2011['Area code'].map(code_mapping).fillna(df_2011['Area code'])
df_2011['Area name 2021'] = df_2011['Area code 2021'].map(name_mapping).fillna(df_2011['Area name'])

num_cols_2011 = [c for c in df_2011.columns if c not in ['Area code', 'Area name', 'Area code 2021', 'Area name 2021']]
df_2011_aligned = df_2011.groupby(['Area code 2021', 'Area name 2021'])[num_cols_2011].sum().reset_index()
df_2011_aligned.rename(columns={'Area code 2021': 'Area code', 'Area name 2021': 'Area name'}, inplace=True)

# 强迫症专属：绝对人数不该有小数点，四舍五入变成整数
for col in num_cols_2011:
    df_2011_aligned[col] = df_2011_aligned[col].round(0)

# ==========================================
# 4. 纯净合并 (此时所有数据都是绝对人数)
# ==========================================
final_mixed = pd.merge(df_2011_aligned, df_2021, on='Area code', how='inner')
if 'Area name_x' in final_mixed.columns:
    final_mixed = final_mixed.rename(columns={'Area name_x': 'Area name'}).drop(columns=['Area name_y'])

# ==========================================
# 5. 🌟 双轨并行：为每列附加百分比兄弟列 🌟
# ==========================================
print("📊 正在保留绝对数量的同时，增设专属的百分比列 (%) ...")
df_final_dual = final_mixed.copy()

# 定位各大主题的分母基数
edu_total_11 = 'All categories: Highest level of qualification'
occ_total_11 = 'All categories: NS-SeC'
health_total_11 = 'General Health: All categories: General health; measures: Value'

edu_total_21 = 'Highest level of qualification: Total: All usual residents aged 16 years and over'
occ_total_21 = 'National Statistics Socio-economic Classification (NS-SEC): Total: All usual residents aged 16 years and over'
health_total_21 = [c for c in final_mixed.columns if 'health: Total' in str(c).lower() or 'general health: all' in str(c).lower()][0]

total_cols_list = [edu_total_11, occ_total_11, health_total_11, edu_total_21, occ_total_21, health_total_21]

# 锁定原有的所有绝对人数列进行遍历（不要去遍历刚刚新生成的百分比列）
original_cols = [col for col in final_mixed.columns if col not in ['Area code', 'Area name', 'date', 'date_x', 'date_y'] + total_cols_list]

for col in original_cols:
    base_col = None
    col_str = str(col).lower()
    
    # 智能分发分母
    if col in df_2011.columns: 
        if 'health' in col_str: base_col = health_total_11
        elif 'occupations' in col_str or 'unemployed' in col_str or 'student' in col_str or 'l' in col_str or 'not classified' in col_str: base_col = occ_total_11
        else: base_col = edu_total_11
    else: 
        if 'health' in col_str: base_col = health_total_21
        elif 'socio' in col_str or 'ns-sec' in col_str or 'managerial' in col_str: base_col = occ_total_21
        else: base_col = edu_total_21
        
    if base_col and base_col in final_mixed.columns:
        # 魔法所在：不覆盖原列，而是创建一个带 " (%)" 的新列！
        pct_col_name = f"{col} (%)"
        df_final_dual[pct_col_name] = (final_mixed[col] / final_mixed[base_col]) * 100
        df_final_dual[pct_col_name] = df_final_dual[pct_col_name].round(1)

# 保存终极表
output_file = 'DUAL_DATA_2011_2021.csv'
df_final_dual.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"🎉 恭喜！完美融合完成！")
print(f"现在表格拥有 {len(df_final_dual.columns)} 列！原名列为绝对人数，带 '(%)' 的列为百分比！")
print(f"💾 数据已安全保存为：{output_file}")

🤝 正在合并 2011 和 2021 的最终数据...


,Area name,Area code,All categories: Highest level of qualification,No qualifications,Highest level of qualification: Level 1 qualifications,Highest level of qualification: Level 2 qualifications_x,Highest level of qualification: Apprenticeship_x,Highest level of qualification: Level 3 qualifications_x,Highest level of qualification: Level 4 qualifications and above_x,Highest level of qualification: Other qualifications_x,...,National Statistics Socio-economic Classification (NS-SEC): L13 Routine occupations,National Statistics Socio-economic Classification (NS-SEC): L14.1 and L14.2 Never worked and long-term unemployed,National Statistics Socio-economic Classification (NS-SEC): L15 Full-time students,date,General health: Total: All usual residents,General health: Very good health,General health: Good health,General health: Fair health,General health: Bad health,General health: Very bad health
0,Adur,E07000223,50579,25.6,15.9,16.5,4.4,11.2,22.0,4.5,...,5265,3632,2555,2021,64544,29272,22573,9108,2800,791
1,Allerdale,E07000026,80155,27.0,13.7,16.2,5.0,11.5,22.8,3.8,...,12639,6949,3122,2021,96154,43162,32866,14357,4498,1271
2,Amber Valley,E07000032,100841,27.0,13.9,15.5,4.6,12.0,23.2,3.9,...,16670,7244,4478,2021,126208,56072,44733,18298,5613,1492
3,Arun,E07000224,126164,24.9,14.2,16.8,4.0,11.6,22.8,5.9,...,15984,9870,6480,2021,164889,71215,60303,24419,7110,1842
4,Ashfield,E07000170,96698,31.4,16.4,16.4,4.3,12.0,15.1,4.4,...,19176,9108,4627,2021,126300,53556,43876,19912,6947,2009
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
320,Wrexham,W06000006,109026,26.7,13.9,16.1,3.9,11.5,22.8,5.0,...,18288,9102,6898,2021,135117,61404,45998,19316,6588,1811
321,Wychavon,E07000238,97039,22.1,13.1,15.6,4.0,11.6,28.7,4.8,...,13023,6396,4845,2021,132492,62836,46380,17163,4829,1284
322,Wyre,E07000128,90467,25.9,13.4,16.6,4.8,12.0,23.0,4.2,...,10324,8107,4833,2021,111946,49623,37251,17326,5902,1844
323,Wyre Forest,E07000239,81190,27.8,14.2,16.1,3.8,11.4,22.1,4.6,...,12129,6612,3826,2021,101607,44004,36337,15024,4818,1424


🎉 终极对齐成功！完美匹配了 325 个平级城市！
💾 完美合并表已保存为: ULTIMATE_2011_2021_Merged.csv

--------------------------------------------------
🧹 正在清理数字格式 (洗除隐藏的逗号，防止计算报错)...
🚀 启动【绝对人数+百分比】双轨制终极引擎...
📊 正在保留绝对数量的同时，增设专属的百分比列 (%) ...
🎉 恭喜！完美融合完成！
现在表格拥有 186 列！原名列为绝对人数，带 '(%)' 的列为百分比！
💾 数据已安全保存为：DUAL_DATA_2011_2021.csv


In [45]:
import pandas as pd

print("🚀 启动特征工程引擎 (Feature Engineering)...")

# 1. 读取双轨制全量数据
df = pd.read_csv('DUAL_DATA_2011_2021.csv')
df_proxy = df[['Area code', 'Area name']].copy()

# ==========================================
# 🎯 第一步：提炼 2011 & 2021 的【绝对人数】
# ==========================================
print("➕ 正在聚合三大宏观维度的绝对人数...")

# --- 📚 教育水平 (Education) ---
# 2011
df_proxy['Edu_Low_2011'] = df['No qualifications']
df_proxy['Edu_High_2011'] = df['Highest level of qualification: Level 4 qualifications and above_x']
df_proxy['Edu_Medium_2011'] = (df['Highest level of qualification: Level 1 qualifications'] + 
                               df['Highest level of qualification: Level 2 qualifications_x'] + 
                               df['Highest level of qualification: Apprenticeship_x'] + 
                               df['Highest level of qualification: Level 3 qualifications_x'] + 
                               df['Highest level of qualification: Other qualifications_x'])

# 2021
df_proxy['Edu_Low_2021'] = df['Highest level of qualification: No qualifications']
df_proxy['Edu_High_2021'] = df['Highest level of qualification: Level 4 qualifications and above_y']
df_proxy['Edu_Medium_2021'] = (df['Highest level of qualification: Level 1 and entry level qualifications'] + 
                               df['Highest level of qualification: Level 2 qualifications_y'] + 
                               df['Highest level of qualification: Apprenticeship_y'] + 
                               df['Highest level of qualification: Level 3 qualifications_y'] + 
                               df['Highest level of qualification: Other qualifications_y'])


# --- 💼 社会等级 (Occupation) ---
# 2011
df_proxy['Occ_Advantaged_2011'] = df['1. Higher managerial, administrative and professional occupations'] + df['2. Lower managerial, administrative and professional occupations']
df_proxy['Occ_Intermediate_2011'] = df['3. Intermediate occupations'] + df['4. Small employers and own account workers'] + df['5. Lower supervisory and technical occupations']
df_proxy['Occ_Disadvantaged_2011'] = df['6. Semi-routine occupations'] + df['7. Routine occupations'] + df['8. Never worked and long-term unemployed']

# 2021
df_proxy['Occ_Advantaged_2021'] = df['National Statistics Socio-economic Classification (NS-SEC): L1, L2 and L3 Higher managerial, administrative and professional occupations'] + df['National Statistics Socio-economic Classification (NS-SEC): L4, L5 and L6 Lower managerial, administrative and professional occupations']
df_proxy['Occ_Intermediate_2021'] = df['National Statistics Socio-economic Classification (NS-SEC): L7 Intermediate occupations'] + df['National Statistics Socio-economic Classification (NS-SEC): L8 and L9 Small employers and own account workers'] + df['National Statistics Socio-economic Classification (NS-SEC): L10 and L11 Lower supervisory and technical occupations']
df_proxy['Occ_Disadvantaged_2021'] = df['National Statistics Socio-economic Classification (NS-SEC): L12 Semi-routine occupations'] + df['National Statistics Socio-economic Classification (NS-SEC): L13 Routine occupations'] + df['National Statistics Socio-economic Classification (NS-SEC): L14.1 and L14.2 Never worked and long-term unemployed']


# --- 🏥 健康水平 (Health) ---
# 2011
df_proxy['Health_Good_2011'] = df['General Health: Very good health; measures: Value'] + df['General Health: Good health; measures: Value']
df_proxy['Health_Poor_2011'] = df['General Health: Fair health; measures: Value'] + df['General Health: Bad health; measures: Value'] + df['General Health: Very bad health; measures: Value']

# 2021
df_proxy['Health_Good_2021'] = df['General health: Very good health'] + df['General health: Good health']
df_proxy['Health_Poor_2021'] = df['General health: Fair health'] + df['General health: Bad health'] + df['General health: Very bad health']


# ==========================================
# 🎯 第二步：提取总基数，并计算【标准化百分比】
# ==========================================
print("➗ 正在将宏观变量标准化为百分比 (%) ...")

# 锁定原表里的各领域总人数
bases = {
    'Edu_2011': df['All categories: Highest level of qualification'],
    'Occ_2011': df['All categories: NS-SeC'],
    'Health_2011': df['General Health: All categories: General health; measures: Value'],
    'Edu_2021': df['Highest level of qualification: Total: All usual residents aged 16 years and over'],
    'Occ_2021': df['National Statistics Socio-economic Classification (NS-SEC): Total: All usual residents aged 16 years and over'],
    'Health_2021': df['General health: Total: All usual residents']
}

# 动态计算上述生成的所有绝对人数列的百分比
proxy_cols = [c for c in df_proxy.columns if c not in ['Area code', 'Area name']]

for col in proxy_cols:
    # 智能匹配对应的分母
    base_key = col.split('_')[0] + '_' + col.split('_')[-1] # 比如 Edu_Low_2011 -> Edu_2011
    
    # 计算百分比并附加新列
    pct_col_name = f"{col} (%)"
    df_proxy[pct_col_name] = (df_proxy[col] / bases[base_key]) * 100
    df_proxy[pct_col_name] = df_proxy[pct_col_name].round(1)

    # 强迫症清理：把绝对人数保留成整数格式，更美观
    df_proxy[col] = df_proxy[col].round(0).astype(int)

# ==========================================
# 🎯 第三步：验收与保存
# ==========================================
output_name = 'PROXY_VARIABLES_2011_2021.csv'
df_proxy.to_csv(output_name, index=False, encoding='utf-8-sig')

print(f"🎉 提取大功告成！")
print(f"📊 你的代理变量集包含了：8 个大类 (3教育 + 3职业 + 2健康) × 2个年份 × 2种形态(人数+比例)")
print(f"💾 高级特征数据集已保存为：{output_name}")

🚀 启动特征工程引擎 (Feature Engineering)...
➕ 正在聚合三大宏观维度的绝对人数...
➗ 正在将宏观变量标准化为百分比 (%) ...
🎉 提取大功告成！
📊 你的代理变量集包含了：8 个大类 (3教育 + 3职业 + 2健康) × 2个年份 × 2种形态(人数+比例)
💾 高级特征数据集已保存为：PROXY_VARIABLES_2011_2021.csv


In [46]:
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

print("🧠 正在启动机器学习引擎：2011年主成分分析 (PCA) ...")

# 1. 读取包含所有特征的终极表
df = pd.read_csv('PROXY_VARIABLES_2011_2021.csv')

# 2. 锁定 2011 年用于降维的 8 个核心百分比特征
features_2011 = [
    'Edu_Low_2011 (%)', 'Edu_Medium_2011 (%)', 'Edu_High_2011 (%)',
    'Occ_Advantaged_2011 (%)', 'Occ_Intermediate_2011 (%)', 'Occ_Disadvantaged_2011 (%)',
    'Health_Good_2011 (%)', 'Health_Poor_2011 (%)'
]

print("⚖️ 正在进行 Z-Score 数据标准化 (Standardization)...")
X_2011 = df[features_2011]

# 初始化标准化器并转换数据 (均值为0，方差为1)
scaler_11 = StandardScaler()
X_scaled_11 = scaler_11.fit_transform(X_2011)

# 3. 执行 PCA 降维
print("📉 正在将 8 维空间压缩为 2 维主成分...")
pca_11 = PCA(n_components=2) # 提取前两个最重要的主成分
pca_scores_11 = pca_11.fit_transform(X_scaled_11)

# 获取解释方差比（看看这两个成分保留了多少原始信息）
variance_ratio = pca_11.explained_variance_ratio_

# 4. 将计算出的 PCA 坐标无缝接入原表
# 我们通常将 PC1 命名为“社会优势指数”，它是区分阶层的核心轴
df['PCA1_2011'] = pca_scores_11[:, 0]
df['PCA2_2011'] = pca_scores_11[:, 1]

# ==========================================
# 5. 打印学术分析报告并保存
# ==========================================
print("\n📊 --- PCA 模型学术报告 ---")
print(f"第一主成分 (PC1) 解释了 {variance_ratio[0]*100:.1f}% 的数据差异！(通常代表整体社会经济优势/劣势)")
print(f"第二主成分 (PC2) 解释了 {variance_ratio[1]*100:.1f}% 的数据差异！")
print(f"前两项累计保留了 {(variance_ratio[0] + variance_ratio[1])*100:.1f}% 的原始信息！(降维非常成功！)")

# 保存包含 PCA 值的新表
output_name = 'PCA_RESULTS_2011.csv'
df.to_csv(output_name, index=False, encoding='utf-8-sig')
print(f"\n🎉 大功告成！带有 2011 年 PCA 坐标系的新表格已保存为：{output_name}")

🧠 正在启动机器学习引擎：2011年主成分分析 (PCA) ...
⚖️ 正在进行 Z-Score 数据标准化 (Standardization)...
📉 正在将 8 维空间压缩为 2 维主成分...

📊 --- PCA 模型学术报告 ---
第一主成分 (PC1) 解释了 68.7% 的数据差异！(通常代表整体社会经济优势/劣势)
第二主成分 (PC2) 解释了 13.6% 的数据差异！
前两项累计保留了 82.2% 的原始信息！(降维非常成功！)

🎉 大功告成！带有 2011 年 PCA 坐标系的新表格已保存为：PCA_RESULTS_2011.csv


In [47]:
import pandas as pd
import numpy as np
from sklearn.linear_model import BayesianRidge

print("🔮 正在启动贝叶斯先验推断引擎 (Bayesian Inference)...")

# 1. 加载我们的终极代理变量表
df = pd.read_csv('PROXY_VARIABLES_2011_2021.csv')

# 2. 自动抓取 2011 和 2021 的百分比特征列
features_2011 = [col for col in df.columns if '2011 (%)' in col]
targets_2021 = [col for col in df.columns if '2021 (%)' in col]

print(f"📥 提取到 {len(features_2011)} 个 2011 年先验特征 (X)。")
print(f"🎯 准备预测 {len(targets_2021)} 个 2021 年目标结果 (Y)。")

# 准备 2011 年的自变量 (X)
X = df[features_2011]

# 建立一个新的 DataFrame，用来存放所有的贝叶斯预测结果
results_df = df[['Area code', 'Area name']].copy()

print("\n⚙️ 正在为每个维度构建贝叶斯概率模型并计算 95% 置信区间...")

# 3. 循环遍历每一个 2021 年的指标，进行贝叶斯拟合和预测
for target in targets_2021:
    y_actual = df[target]
    
    # 初始化贝叶斯岭回归模型
    # 它会自动计算参数的后验分布 (Posterior Distribution)
    bayes_model = BayesianRidge(compute_score=True)
    bayes_model.fit(X, y_actual)
    
    # 🌟 核心魔法：不仅输出预测值 (y_pred)，还要输出标准差 (y_std) 以代表不确定性！
    y_pred, y_std = bayes_model.predict(X, return_std=True)
    
    # 计算 95% 贝叶斯可信区间 (Credible Interval)
    # 在正态分布假设下，95% 的区间大约是 预测值 ± 1.96 * 标准差
    ci_lower = y_pred - (1.96 * y_std)
    ci_upper = y_pred + (1.96 * y_std)
    
    # 提取变量的干净名字 (比如把 'Edu_High_2021 (%)' 变成 'Edu_High')
    base_name = target.replace('_2021 (%)', '')
    
    # 将结果写入新表，并保留一位小数
    results_df[f'{base_name}_Actual_21'] = y_actual.round(1)                # 真实值
    results_df[f'{base_name}_Bayes_Pred'] = np.round(y_pred, 1)             # 贝叶斯预测中值
    results_df[f'{base_name}_CI_Lower'] = np.round(ci_lower, 1)             # 95%区间下限
    results_df[f'{base_name}_CI_Upper'] = np.round(ci_upper, 1)             # 95%区间上限

# ==========================================
# 4. 打印验收报告并保存
# ==========================================
print("\n✅ 贝叶斯网络推断完成！抽查第一行城市的【高等教育比例】预测结果：")
print(f"真实值:   {results_df['Edu_High_Actual_21'].iloc[0]} %")
print(f"预测中值: {results_df['Edu_High_Bayes_Pred'].iloc[0]} %")
print(f"95%区间:  [{results_df['Edu_High_CI_Lower'].iloc[0]} %, {results_df['Edu_High_CI_Upper'].iloc[0]} %]")

# 导出极其壮观的贝叶斯预测表
output_name = 'BAYESIAN_PREDICTIONS_2021.csv'
results_df.to_csv(output_name, index=False, encoding='utf-8-sig')
print(f"\n💾 包含所有维度 95% 置信区间的贝叶斯大表已保存为：{output_name}")

🔮 正在启动贝叶斯先验推断引擎 (Bayesian Inference)...
📥 提取到 8 个 2011 年先验特征 (X)。
🎯 准备预测 8 个 2021 年目标结果 (Y)。

⚙️ 正在为每个维度构建贝叶斯概率模型并计算 95% 置信区间...

✅ 贝叶斯网络推断完成！抽查第一行城市的【高等教育比例】预测结果：
真实值:   24.8 %
预测中值: 23.4 %
95%区间:  [-30.6 %, 77.5 %]

💾 包含所有维度 95% 置信区间的贝叶斯大表已保存为：BAYESIAN_PREDICTIONS_2021.csv


In [48]:
import pandas as pd
import numpy as np
from sklearn.linear_model import BayesianRidge
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

print("🚀 正在构建【PCA降维 + 贝叶斯推断】大一统矩阵 (Master Table)...")

# 1. 加载包含特征数据的原始代理变量表
df = pd.read_csv('PROXY_VARIABLES_2011_2021.csv')

# 自动匹配带百分号的特征列
features_2011 = [col for col in df.columns if '2011' in col and '(%)' in col]
targets_2021 = [col for col in df.columns if '2021' in col and '(%)' in col]

X = df[features_2011]
# 创建基础汇总表，保留城市代码和名称
results_df = df[['Area code', 'Area name']].copy()


# ==========================================
# 2. 计算 2011 年主成分 (PCA) 并加入总表
# ==========================================
print("📉 正在执行 2011 年数据标准化与 PCA 降维...")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=2)
pca_scores = pca.fit_transform(X_scaled)

# 将 PCA 坐标保留 3 位小数，加入汇总表
results_df['PCA1_2011_Social_Advantage'] = np.round(pca_scores[:, 0], 3)
results_df['PCA2_2011_Secondary_Axis'] = np.round(pca_scores[:, 1], 3)

variance_ratio = pca.explained_variance_ratio_
print(f"   -> PC1 解释了 {variance_ratio[0]*100:.1f}% 的方差，PC2 解释了 {variance_ratio[1]*100:.1f}% 的方差。")


# ==========================================
# 3. 执行 2021 年贝叶斯推断并加入总表
# ==========================================
print("🔮 正在拟合贝叶斯先验网络并提取 95% 置信区间...")

for target in targets_2021:
    y_actual = df[target]
    
    # 训练贝叶斯模型并提取预测值及标准差
    bayes_model = BayesianRidge(compute_score=True)
    bayes_model.fit(X, y_actual)
    y_pred, y_std = bayes_model.predict(X, return_std=True)
    
    # 计算 95% 置信区间
    ci_lower = y_pred - (1.96 * y_std)
    ci_upper = y_pred + (1.96 * y_std)
    
    # 清理列名以保持表格整洁 (去掉多余的年份和符号)
    base_name = target.split('_2021')[0] if '_2021' in target else target.replace('2021_', '').replace(' (%)', '')
    
    # 拼接到结果表
    results_df[f'{base_name}_Actual_21'] = y_actual.round(1)
    results_df[f'{base_name}_Bayes_Pred'] = np.round(y_pred, 1)
    results_df[f'{base_name}_CI_Lower'] = np.round(ci_lower, 1)
    results_df[f'{base_name}_CI_Upper'] = np.round(ci_upper, 1)


# ==========================================
# 4. 保存终极大宽表
# ==========================================
output_name = 'MASTER_RESULTS_2011_2021.csv'
results_df.to_csv(output_name, index=False, encoding='utf-8-sig')

print(f"\n🎉 大满贯！所有数据已完美融合！")
print(f"💾 包含 [城市信息 + PCA坐标 + 贝叶斯全套预测] 的终极表格已保存为：{output_name}")
print(f"表格维度: {results_df.shape[0]} 个城市 × {results_df.shape[1]} 列核心数据。直接导入 Tableau 起飞吧！")

🚀 正在构建【PCA降维 + 贝叶斯推断】大一统矩阵 (Master Table)...
📉 正在执行 2011 年数据标准化与 PCA 降维...
   -> PC1 解释了 68.7% 的方差，PC2 解释了 13.6% 的方差。
🔮 正在拟合贝叶斯先验网络并提取 95% 置信区间...

🎉 大满贯！所有数据已完美融合！
💾 包含 [城市信息 + PCA坐标 + 贝叶斯全套预测] 的终极表格已保存为：MASTER_RESULTS_2011_2021.csv
表格维度: 325 个城市 × 36 列核心数据。直接导入 Tableau 起飞吧！


In [51]:
import pandas as pd

print("🚀 正在将【绝对人口】与【总人口基数】汇入终极大师表...")

# ==========================================
# 1. 读取之前生成的三张核心表
# ==========================================
# 包含 PCA 和 贝叶斯预测的宽表
df_master = pd.read_csv('MASTER_RESULTS_2011_2021.csv')
# 包含各阶层绝对人数的代理变量表
df_proxy = pd.read_csv('PROXY_VARIABLES_2011_2021.csv')
# 包含各维度原始总基数的双轨表
df_dual = pd.read_csv('DUAL_DATA_2011_2021.csv')


# ==========================================
# 2. 从 DUAL 表中提取各大类的“总人口基数”并规范命名
# ==========================================
# 锁定原始的总计列名
edu_total_11_col = 'All categories: Highest level of qualification'
occ_total_11_col = 'All categories: NS-SeC'
health_total_11_col = 'General Health: All categories: General health; measures: Value'

edu_total_21_col = 'Highest level of qualification: Total: All usual residents aged 16 years and over'
occ_total_21_col = 'National Statistics Socio-economic Classification (NS-SEC): Total: All usual residents aged 16 years and over'
health_total_21_col = [c for c in df_dual.columns if 'health: Total' in str(c).lower() or 'general health: all' in str(c).lower()][0]

# 提取并重命名，加上明确的 Total 标识
df_totals = df_dual[['Area code']].copy()
df_totals['Total_Edu_2011'] = df_dual[edu_total_11_col]
df_totals['Total_Occ_2011'] = df_dual[occ_total_11_col]
df_totals['Total_Health_2011'] = df_dual[health_total_11_col]

df_totals['Total_Edu_2021'] = df_dual[edu_total_21_col]
df_totals['Total_Occ_2021'] = df_dual[occ_total_21_col]
df_totals['Total_Health_2021'] = df_dual[health_total_21_col]


# ==========================================
# 3. 从 PROXY 表中提取各变量的“实际绝对人数”
# ==========================================
# 自动抓取不带 (%) 的列，这些就是我们之前算好的绝对人数
# 保留 Area code 用于拼接
abs_proxy_cols = [c for c in df_proxy.columns if '(%)' not in c and c != 'Area name']
df_abs_proxies = df_proxy[abs_proxy_cols]


# ==========================================
# 4. 完美拼接 (Left Join)
# ==========================================
print("🧩 正在执行高精度表拼接 (Left Join)...")

# 先把 Master 表和 总基数表 拼起来
df_final = pd.merge(df_master, df_totals, on='Area code', how='left')

# 再把 各维度绝对人数 拼进去
df_final = pd.merge(df_final, df_abs_proxies, on='Area code', how='left')


# ==========================================
# 5. 保存最终形态
# ==========================================
output_name = 'ULTIMATE_MASTER_WITH_POPULATION.csv'
df_final.to_csv(output_name, index=False, encoding='utf-8-sig')

print(f"\n🎉 整合大获成功！")
print(f"💾 新的六边形战士表格已保存为：{output_name}")
print(f"📈 表格维度: {df_final.shape[0]} 个城市 × {df_final.shape[1]} 列")
print("✅ 新增内容：2011与2021的6个总人口基数，以及所有细分特征的具体人数！")

🚀 正在将【绝对人口】与【总人口基数】汇入终极大师表...
🧩 正在执行高精度表拼接 (Left Join)...

🎉 整合大获成功！
💾 新的六边形战士表格已保存为：ULTIMATE_MASTER_WITH_POPULATION.csv
📈 表格维度: 325 个城市 × 58 列
✅ 新增内容：2011与2021的6个总人口基数，以及所有细分特征的具体人数！


In [54]:
import pandas as pd
import numpy as np

print("🪄 正在按照顶级学术规范重构并排版终极大师表...")

# 1. 读取完整表
df = pd.read_csv('ULTIMATE_MASTER_WITH_POPULATION.csv')
df_perfect = pd.DataFrame()

# ==========================================
# 📍 A. 基础地理信息
# ==========================================
df_perfect['Area_Code'] = df['Area code']
df_perfect['Area_Name'] = df['Area name']

# ==========================================
# 📊 B. 2011 年核心三联组 (Total -> Abs -> Pct)
# ==========================================
# 1. 教育 (高等教育)
df_perfect['Total_Edu_2011'] = df['Total_Edu_2011']
df_perfect['EduHigh_Abs_2011'] = df['Edu_High_2011']
df_perfect['Pct_EduHigh_2011'] = (df['Edu_High_2011'] / df['Total_Edu_2011'] * 100).round(1)

# 2. 职业 (优势阶层)
df_perfect['Total_Occ_2011'] = df['Total_Occ_2011']
df_perfect['OccAdvantaged_Abs_2011'] = df['Occ_Advantaged_2011']
df_perfect['Pct_OccAdvantaged_2011'] = (df['Occ_Advantaged_2011'] / df['Total_Occ_2011'] * 100).round(1)

# 3. 健康 (健康良好)
df_perfect['Total_Health_2011'] = df['Total_Health_2011']
df_perfect['HealthGood_Abs_2011'] = df['Health_Good_2011']
df_perfect['Pct_HealthGood_2011'] = (df['Health_Good_2011'] / df['Total_Health_2011'] * 100).round(1)


# ==========================================
# 📈 C. 2021 年核心三联组
# ==========================================
# 1. 教育
df_perfect['Total_Edu_2021'] = df['Total_Edu_2021']
df_perfect['EduHigh_Abs_2021'] = df['Edu_High_2021']
df_perfect['Pct_EduHigh_2021'] = (df['Edu_High_2021'] / df['Total_Edu_2021'] * 100).round(1)

# 2. 职业
df_perfect['Total_Occ_2021'] = df['Total_Occ_2021']
df_perfect['OccAdvantaged_Abs_2021'] = df['Occ_Advantaged_2021']
df_perfect['Pct_OccAdvantaged_2021'] = (df['Occ_Advantaged_2021'] / df['Total_Occ_2021'] * 100).round(1)

# 3. 健康
df_perfect['Total_Health_2021'] = df['Total_Health_2021']
df_perfect['HealthGood_Abs_2021'] = df['Health_Good_2021']
df_perfect['Pct_HealthGood_2021'] = (df['Health_Good_2021'] / df['Total_Health_2021'] * 100).round(1)


# ==========================================
# 🔮 D. 贝叶斯预测与误差分析
# ==========================================
# 高等教育
df_perfect['Bayes_Pred_EduHigh_2021'] = df['Edu_High_Bayes_Pred']
df_perfect['Error_EduHigh_2021'] = (df_perfect['Pct_EduHigh_2021'] - df['Edu_High_Bayes_Pred']).round(1)
df_perfect['Lower_EduHigh_2021'] = df['Edu_High_CI_Lower']
df_perfect['Upper_EduHigh_2021'] = df['Edu_High_CI_Upper']

# 优势职业
df_perfect['Bayes_Pred_OccAdvantaged_2021'] = df['Occ_Advantaged_Bayes_Pred']
df_perfect['Error_OccAdvantaged_2021'] = (df_perfect['Pct_OccAdvantaged_2021'] - df['Occ_Advantaged_Bayes_Pred']).round(1)
df_perfect['Lower_OccAdvantaged_2021'] = df['Occ_Advantaged_CI_Lower']
df_perfect['Upper_OccAdvantaged_2021'] = df['Occ_Advantaged_CI_Upper']

# 良好健康
df_perfect['Bayes_Pred_HealthGood_2021'] = df['Health_Good_Bayes_Pred']
df_perfect['Error_HealthGood_2021'] = (df_perfect['Pct_HealthGood_2021'] - df['Health_Good_Bayes_Pred']).round(1)
df_perfect['Lower_HealthGood_2021'] = df['Health_Good_CI_Lower']
df_perfect['Upper_HealthGood_2021'] = df['Health_Good_CI_Upper']


# ==========================================
# 📉 E. 降维坐标系 (PCA)
# ==========================================
df_perfect['PCA_1'] = df['PCA1_2011_Social_Advantage']
df_perfect['PCA_2'] = df['PCA2_2011_Secondary_Axis']


# ==========================================
# 💾 保存神仙排版表格
# ==========================================
output_name = 'FINAL_FORMATTED_TABLE_2011_2021.csv'
df_perfect.to_csv(output_name, index=False, encoding='utf-8-sig')

print(f"🎉 完美重组成功！真正的最终版！")
print(f"💾 表格已保存为：{output_name}")
print(f"快去看看 Error 列吧，正数代表超出预期，负数代表低于预期！")

🪄 正在按照顶级学术规范重构并排版终极大师表...
🎉 完美重组成功！真正的最终版！
💾 表格已保存为：FINAL_FORMATTED_TABLE_2011_2021.csv
快去看看 Error 列吧，正数代表超出预期，负数代表低于预期！


In [27]:
import pandas as pd
import numpy as np

# 1. 读取最原始的源文件
df_2011 = pd.read_csv('Census_2011_Cleaned_Original.csv')
df_2021 = pd.read_csv('Census_2021_Cleaned_Original.csv')
df_lookup = pd.read_csv('Local_Authority_District_(2011)_to_Local_Authority_District_(2021)_Lookup_for_England_and_Wales.csv')

print("🛠️ 正在重构严谨的 [绝对人口 + 百分比] 终极总表...")

# ==========================================
# 阶段 1: 处理 2011 年数据 (反推绝对人数)
# ==========================================
# 提取 2011 总人口基数 (去除千分位逗号并转为数字)
df_2011['2011_Pop_Total'] = df_2011['All categories: Highest level of qualification'].astype(str).str.replace(',', '').astype(float)

# 精确聚合 2011 年的代理变量 (此时还是百分比)
df_2011['pct_Class_Advantaged'] = df_2011['1. Higher managerial, administrative and professional occupations'] + df_2011['2. Lower managerial, administrative and professional occupations']
df_2011['pct_Class_Intermediate'] = df_2011['3. Intermediate occupations'] + df_2011['5. Lower supervisory and technical occupations']
df_2011['pct_Class_Disadvantaged'] = df_2011['6. Semi-routine occupations'] + df_2011['7. Routine occupations'] + df_2011['8. Never worked and long-term unemployed']
df_2011['pct_Edu_High'] = df_2011['Highest level of qualification: Level 4 qualifications and above']
df_2011['pct_Edu_Mid'] = df_2011['Highest level of qualification: Level 1 qualifications'] + df_2011['Highest level of qualification: Level 2 qualifications'] + df_2011['Highest level of qualification: Apprenticeship'] + df_2011['Highest level of qualification: Level 3 qualifications'] + df_2011['Highest level of qualification: Other qualifications']
df_2011['pct_Edu_Low'] = df_2011['No qualifications']

# 核心：将百分比转换为 2011 年的真实绝对人数！
proxies = ['Class_Advantaged', 'Class_Intermediate', 'Class_Disadvantaged', 'Edu_High', 'Edu_Mid', 'Edu_Low']
for p in proxies:
    df_2011[f'2011_{p}_Count'] = (df_2011[f'pct_{p}'] / 100) * df_2011['2011_Pop_Total']

# 地理映射与按 2021 年版图合并 (这次合并的是绝对人数！)
df_2011['Cleaned_Name'] = df_2011['Area name'].astype(str).str.strip().str.replace(r' UA$', '', regex=True).str.replace('’', "'").replace({'Rhondda Cynon Taf': 'Rhondda Cynon Taff'})
mapping = dict(zip(df_lookup['LAD11NM'], df_lookup['LAD21NM']))
df_2011['Area name 2021'] = df_2011['Cleaned_Name'].map(mapping).fillna(df_2011['Cleaned_Name'])

cols_to_sum = ['2011_Pop_Total'] + [f'2011_{p}_Count' for p in proxies]
df_2011_aligned = df_2011.groupby('Area name 2021')[cols_to_sum].sum().reset_index().rename(columns={'Area name 2021': 'Area name'})

# 核心：用合并后的人数，重新算出 2011 年完美的科学百分比！
for p in proxies:
    df_2011_aligned[f'2011_{p}_Pct'] = (df_2011_aligned[f'2011_{p}_Count'] / df_2011_aligned['2011_Pop_Total']) * 100
    df_2011_aligned[f'2011_{p}_Pct'] = df_2011_aligned[f'2011_{p}_Pct'].round(1)
    df_2011_aligned[f'2011_{p}_Count'] = df_2011_aligned[f'2011_{p}_Count'].round(0).astype(int) # 人数转为整数

# ==========================================
# 阶段 2: 处理 2021 年数据 (原生绝对人数转百分比)
# ==========================================
df_2021['2021_Pop_Total'] = df_2021['Highest level of qualification: Total: All usual residents aged 16 years and over']

# 聚合 2021 绝对人数
df_2021['2021_Class_Advantaged_Count'] = df_2021['National Statistics Socio-economic Classification (NS-SEC): L1, L2 and L3 Higher managerial, administrative and professional occupations'] + df_2021['National Statistics Socio-economic Classification (NS-SEC): L4, L5 and L6 Lower managerial, administrative and professional occupations']
df_2021['2021_Class_Intermediate_Count'] = df_2021['National Statistics Socio-economic Classification (NS-SEC): L7 Intermediate occupations'] + df_2021['National Statistics Socio-economic Classification (NS-SEC): L10 and L11 Lower supervisory and technical occupations']
df_2021['2021_Class_Disadvantaged_Count'] = df_2021['National Statistics Socio-economic Classification (NS-SEC): L12 Semi-routine occupations'] + df_2021['National Statistics Socio-economic Classification (NS-SEC): L13 Routine occupations'] + df_2021['National Statistics Socio-economic Classification (NS-SEC): L14.1 and L14.2 Never worked and long-term unemployed']
df_2021['2021_Edu_High_Count'] = df_2021['Highest level of qualification: Level 4 qualifications and above']
df_2021['2021_Edu_Mid_Count'] = df_2021['Highest level of qualification: Level 1 and entry level qualifications'] + df_2021['Highest level of qualification: Level 2 qualifications'] + df_2021['Highest level of qualification: Apprenticeship'] + df_2021['Highest level of qualification: Level 3 qualifications'] + df_2021['Highest level of qualification: Other qualifications']
df_2021['2021_Edu_Low_Count'] = df_2021['Highest level of qualification: No qualifications']

# 计算 2021 百分比
for p in proxies:
    df_2021[f'2021_{p}_Pct'] = (df_2021[f'2021_{p}_Count'] / df_2021['2021_Pop_Total']) * 100
    df_2021[f'2021_{p}_Pct'] = df_2021[f'2021_{p}_Pct'].round(1)

cols_2021_keep = ['Area name', '2021_Pop_Total']
for p in proxies:
    cols_2021_keep.extend([f'2021_{p}_Count', f'2021_{p}_Pct'])

df_2021_final = df_2021[cols_2021_keep]

# ==========================================
# 阶段 3: 世纪大合并
# ==========================================
final_master = pd.merge(df_2011_aligned, df_2021_final, on='Area name', how='inner')

# 保存这套无懈可击的数据集！
final_master.to_csv('MASTER_COUNTS_AND_PCT_2011_2021.csv', index=False, encoding='utf-8-sig')

# 验收之前爆炸的白金汉郡数据
print(f"✅ 漏洞修复成功！Buckinghamshire 2011 精英阶层真实比例重归合理: {final_master[final_master['Area name']=='Buckinghamshire']['2011_Class_Advantaged_Pct'].iloc[0]} %")
print(f"🎉 终极 Master 表格已保存为: MASTER_COUNTS_AND_PCT_2011_2021.csv (共 {len(final_master.columns)} 列，包含一切你所需的信息！)")

🛠️ 正在重构严谨的 [绝对人口 + 百分比] 终极总表...
✅ 漏洞修复成功！Buckinghamshire 2011 精英阶层真实比例重归合理: 51.4 %
🎉 终极 Master 表格已保存为: MASTER_COUNTS_AND_PCT_2011_2021.csv (共 27 列，包含一切你所需的信息！)


In [28]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# 1. 读取我们堪称艺术品的 Master 表
df_master = pd.read_csv('MASTER_COUNTS_AND_PCT_2011_2021.csv')

# 2. 精准提取我们要降维的百分比特征列
cols_2011_pct = [
    '2011_Class_Advantaged_Pct', '2011_Class_Intermediate_Pct', '2011_Class_Disadvantaged_Pct',
    '2011_Edu_High_Pct', '2011_Edu_Mid_Pct', '2011_Edu_Low_Pct'
]

cols_2021_pct = [
    '2021_Class_Advantaged_Pct', '2021_Class_Intermediate_Pct', '2021_Class_Disadvantaged_Pct',
    '2021_Edu_High_Pct', '2021_Edu_Mid_Pct', '2021_Edu_Low_Pct'
]

print("🚀 正在启动标准化与 PCA 降维引擎...")

# 创建一个干净的新表来存放城市的二维坐标
df_pca = df_master[['Area name']].copy()

# ==========================================
# 提取 2011 年的主成分坐标
# ==========================================
# 标准化 (Z-score)：让所有阶层特征站上同一起跑线
scaler_2011 = StandardScaler()
X_2011_scaled = scaler_2011.fit_transform(df_master[cols_2011_pct])

# PCA 降维：将 6 个特征压缩为 2 个超级维度
pca_2011 = PCA(n_components=2)
pca_2011_scores = pca_2011.fit_transform(X_2011_scaled)

# 记录坐标和信息保留率
df_pca['2011_PC1'] = pca_2011_scores[:, 0]
df_pca['2011_PC2'] = pca_2011_scores[:, 1]
var_2011 = pca_2011.explained_variance_ratio_.sum() * 100

# ==========================================
# 提取 2021 年的主成分坐标
# ==========================================
# 标准化 (Z-score)
scaler_2021 = StandardScaler()
X_2021_scaled = scaler_2021.fit_transform(df_master[cols_2021_pct])

# PCA 降维
pca_2021 = PCA(n_components=2)
pca_2021_scores = pca_2021.fit_transform(X_2021_scaled)

# 记录坐标和信息保留率
df_pca['2021_PC1'] = pca_2021_scores[:, 0]
df_pca['2021_PC2'] = pca_2021_scores[:, 1]
var_2021 = pca_2021.explained_variance_ratio_.sum() * 100

# ==========================================
# 打印分析报告并保存
# ==========================================
print(f"\n📊 降维成果：")
print(f"2011年：仅用 2 个维度就保留了原始 6 个变量 {var_2011:.1f}% 的信息量！")
print(f"2021年：仅用 2 个维度就保留了原始 6 个变量 {var_2021:.1f}% 的信息量！")

print("\n🧠 揭秘 PC1 和 PC2 到底代表什么 (以2021年为例)：")
weights_2021 = pd.DataFrame(
    pca_2021.components_.T, 
    columns=['PC1 权重', 'PC2 权重'], 
    index=[c.replace('2021_', '').replace('_Pct', '') for c in cols_2021_pct]
)
print(weights_2021.round(2))

# 保存成可以直接做 KMeans 和画散点图的文件
df_pca = df_pca.round(3)
df_pca.to_csv('FINAL_PCA_SCORES_2011_2021.csv', index=False, encoding='utf-8-sig')
print("\n🎉 大功告成！全英国 331 个城市十年前后的二维坐标系已保存为: FINAL_PCA_SCORES_2011_2021.csv")

🚀 正在启动标准化与 PCA 降维引擎...

📊 降维成果：
2011年：仅用 2 个维度就保留了原始 6 个变量 92.8% 的信息量！
2021年：仅用 2 个维度就保留了原始 6 个变量 94.9% 的信息量！

🧠 揭秘 PC1 和 PC2 到底代表什么 (以2021年为例)：
                     PC1 权重  PC2 权重
Class_Advantaged       0.42    0.33
Class_Intermediate    -0.34    0.59
Class_Disadvantaged   -0.42   -0.38
Edu_High               0.46   -0.15
Edu_Mid               -0.39    0.48
Edu_Low               -0.41   -0.37

🎉 大功告成！全英国 331 个城市十年前后的二维坐标系已保存为: FINAL_PCA_SCORES_2011_2021.csv


In [30]:
import pandas as pd

print("📦 正在打包终极 Excel 数据库...")

# 1. 读取之前生成的两份完美数据
df_master = pd.read_csv('MASTER_COUNTS_AND_PCT_2011_2021.csv')
df_pca = pd.read_csv('FINAL_PCA_SCORES_2011_2021.csv')

# 2. 通过“地区名称”进行内连接合并
df_combined = pd.merge(df_master, df_pca, on='Area name', how='inner')

# ==========================================
# 3. 强迫症福音：重新排版列的顺序
# ==========================================
# 把地名放第一列
final_cols = ['Area name']

# --- 2011 年数据板块 ---
# 先放总人口和最重要的 PCA 坐标
final_cols.extend(['2011_Pop_Total', '2011_PC1', '2011_PC2'])
# 再放 2011 年的各种比例和人数
cols_2011_rest = [c for c in df_combined.columns if '2011' in c and c not in final_cols]
final_cols.extend(sorted(cols_2011_rest))

# --- 2021 年数据板块 ---
# 先放总人口和最重要的 PCA 坐标
final_cols.extend(['2021_Pop_Total', '2021_PC1', '2021_PC2'])
# 再放 2021 年的各种比例和人数
cols_2021_rest = [c for c in df_combined.columns if '2021' in c and c not in final_cols]
final_cols.extend(sorted(cols_2021_rest))

# 应用新的列顺序
df_final = df_combined[final_cols]

# ==========================================
# 4. 导出为高清 Excel 表格 (.xlsx)
# ==========================================
excel_name = 'ULTIMATE_DATABASE_2011_2021.xlsx'

# 使用 ExcelWriter 可以让你直接生成 xlsx 格式
try:
    df_final.to_excel(excel_name, index=False, sheet_name='Decade_Comparison')
    print(f"🎉 大功告成！全景 Excel 数据表已生成: {excel_name}")
    print("👉 你现在可以直接双击打开这个 Excel 文件，查看包含 PCA 坐标的完整数据了！")
except ModuleNotFoundError:
    # 如果你的环境没安装 openpyxl (导出 Excel 需要的底层包)，我们会自动降级保存为 CSV
    print("⚠️ 提示：你的环境未安装 Excel 导出引擎 (openpyxl)。")
    print("正在为你保存为终极 CSV 版 (你可以用 Excel 打开它并另存为 xlsx)...")
    csv_fallback = 'ULTIMATE_DATABASE_2011_2021_Final.csv'
    df_final.to_csv(csv_fallback, index=False, encoding='utf-8-sig')
    print(f"💾 已保存为: {csv_fallback}")

📦 正在打包终极 Excel 数据库...
🎉 大功告成！全景 Excel 数据表已生成: ULTIMATE_DATABASE_2011_2021.xlsx
👉 你现在可以直接双击打开这个 Excel 文件，查看包含 PCA 坐标的完整数据了！


In [32]:
import pandas as pd
import numpy as np
from sklearn.linear_model import BayesianRidge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error

print("🔮 正在启动贝叶斯预测引擎...")

# 1. 读取我们的 Master 终极表
df = pd.read_csv('MASTER_COUNTS_AND_PCT_2011_2021.csv')

# 2. 定义自变量 X (2011年的6个代理变量) 和 因变量 Y (2021年的对应变量)
cols_2011 = [
    '2011_Class_Advantaged_Pct', '2011_Class_Intermediate_Pct', '2011_Class_Disadvantaged_Pct',
    '2011_Edu_High_Pct', '2011_Edu_Mid_Pct', '2011_Edu_Low_Pct'
]
cols_2021 = [
    '2021_Class_Advantaged_Pct', '2021_Class_Intermediate_Pct', '2021_Class_Disadvantaged_Pct',
    '2021_Edu_High_Pct', '2021_Edu_Mid_Pct', '2021_Edu_Low_Pct'
]

# 提取 X 并进行标准化 (贝叶斯模型在标准化数据上表现更好)
X = df[cols_2011]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 创建一个新表，保存预测结果
df_predictions = df[['Area name']].copy()

print("\n📊 贝叶斯模型拟合报告：")

# 3. 针对 2021 年的 6 个目标变量，分别训练贝叶斯模型
for col_y in cols_2021:
    y_true = df[col_y]
    
    # 初始化贝叶斯岭回归模型
    bayesian_model = BayesianRidge()
    
    # 拟合模型
    bayesian_model.fit(X_scaled, y_true)
    
    # 预测并同时获取不确定性 (return_std=True 是贝叶斯的核心魔法！)
    y_pred, y_std = bayesian_model.predict(X_scaled, return_std=True)
    
    # 评估模型准确度
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    
    # 打印每个变量的预测成绩
    target_name = col_y.replace('2021_', '').replace('_Pct', '')
    print(f"🎯 预测目标: {target_name:<20} | 解释度(R²): {r2:.3f} | 平均误差: ±{mae:.2f}%")
    
    # 将预测值和标准差保存到表格中
    df_predictions[f'Predicted_{target_name}_Pct'] = np.round(y_pred, 1)
    df_predictions[f'Uncertainty_Std_{target_name}'] = np.round(y_std, 2)
    # 把真实的2021年数据也放进来方便对比
    df_predictions[f'Actual_2021_{target_name}_Pct'] = y_true

# 4. 算出每个城市预测的“绝对误差” (真实值 - 预测值的绝对差)
# 这能帮我们找出哪些城市发生了“不可预测的突变”！
for col_y in cols_2021:
    target_name = col_y.replace('2021_', '').replace('_Pct', '')
    df_predictions[f'Error_{target_name}'] = np.abs(df_predictions[f'Actual_2021_{target_name}_Pct'] - df_predictions[f'Predicted_{target_name}_Pct'])

# 5. 导出结果
df_predictions.to_csv('BAYESIAN_PREDICTIONS_2021.csv', index=False, encoding='utf-8-sig')
print("\n🎉 贝叶斯预测完成！结果已保存为: BAYESIAN_PREDICTIONS_2021.csv")

# 抽查第一个城市 (比如 Adur) 的高学历预测情况
print("\n👀 抽查第一个城市的高学历 (Edu_High) 预测：")
sample = df_predictions.iloc[0]
print(f"城市: {sample['Area name']}")
print(f"真实 2021 比例: {sample['Actual_2021_Edu_High_Pct']}%")
print(f"贝叶斯预测比例: {sample['Predicted_Edu_High_Pct']}% (置信波动: ±{sample['Uncertainty_Std_Edu_High']}%)")

🔮 正在启动贝叶斯预测引擎...

📊 贝叶斯模型拟合报告：
🎯 预测目标: Class_Advantaged     | 解释度(R²): 0.974 | 平均误差: ±0.81%
🎯 预测目标: Class_Intermediate   | 解释度(R²): 0.933 | 平均误差: ±0.44%
🎯 预测目标: Class_Disadvantaged  | 解释度(R²): 0.974 | 平均误差: ±0.84%
🎯 预测目标: Edu_High             | 解释度(R²): 0.975 | 平均误差: ±0.99%
🎯 预测目标: Edu_Mid              | 解释度(R²): 0.910 | 平均误差: ±1.34%
🎯 预测目标: Edu_Low              | 解释度(R²): 0.929 | 平均误差: ±0.78%

🎉 贝叶斯预测完成！结果已保存为: BAYESIAN_PREDICTIONS_2021.csv

👀 抽查第一个城市的高学历 (Edu_High) 预测：
城市: Adur
真实 2021 比例: 27.9%
贝叶斯预测比例: 27.7% (置信波动: ±1.36%)


In [33]:
import pandas as pd

print("🔗 正在将 PCA 坐标融合进贝叶斯预测表...")

# 1. 读取我们刚生成的贝叶斯预测表和之前的 PCA 坐标表
df_bayes = pd.read_csv('BAYESIAN_PREDICTIONS_2021.csv')
df_pca = pd.read_csv('FINAL_PCA_SCORES_2011_2021.csv')

# 2. 通过“地区名称(Area name)”进行无缝合并
df_combined = pd.merge(df_bayes, df_pca, on='Area name', how='inner')

# 3. 优化列的排版顺序 (把地名和 PCA 坐标放在最前面)
cols = df_combined.columns.tolist()

# 提取我们要前置的 PCA 列名
pca_cols = ['2011_PC1', '2011_PC2', '2021_PC1', '2021_PC2']

# 从原列表中移除这些列
for col in pca_cols:
    cols.remove(col)
cols.remove('Area name')

# 重新组装：地名 -> PCA 坐标 -> 贝叶斯预测和误差数据
new_order = ['Area name'] + pca_cols + cols
df_final = df_combined[new_order]

# 4. 导出最终的融合表
file_name = 'BAYESIAN_PREDICTIONS_WITH_PCA.csv'
df_final.to_csv(file_name, index=False, encoding='utf-8-sig')

# 顺手存一个 Excel 版本方便你肉眼查看和汇报
try:
    df_final.to_excel('BAYESIAN_PREDICTIONS_WITH_PCA.xlsx', index=False)
    print("✅ 融合成功！同时为你生成了 CSV 和 Excel 双版本！")
except:
    print("✅ 融合成功！已保存为纯净的 CSV 文件。")

print(f"💾 数据集已更新为: {file_name}")

# 预览一下 Adur 融合后的模样
print("\n👀 抽查 Adur 地区融合后的前 7 列数据：")
print(df_final.iloc[0, :7])

🔗 正在将 PCA 坐标融合进贝叶斯预测表...
✅ 融合成功！同时为你生成了 CSV 和 Excel 双版本！
💾 数据集已更新为: BAYESIAN_PREDICTIONS_WITH_PCA.csv

👀 抽查 Adur 地区融合后的前 7 列数据：
Area name                            Adur
2011_PC1                           -1.015
2011_PC2                            0.721
2021_PC1                           -0.917
2021_PC2                            0.779
Predicted_Class_Advantaged_Pct       33.1
Uncertainty_Std_Class_Advantaged     1.12
Name: 0, dtype: object


In [34]:
import pandas as pd

print("🌌 正在启动宇宙级大融合：汇集所有绝对人口、百分比、PCA与贝叶斯预测...")

# 1. 读入包含所有【绝对人数】和【百分比】的基础母表
df_master = pd.read_csv('MASTER_COUNTS_AND_PCT_2011_2021.csv')

# 2. 读入包含【PCA坐标】和【贝叶斯预测/误差】的分析表
df_bayes_pca = pd.read_csv('BAYESIAN_PREDICTIONS_WITH_PCA.csv')

# 3. 清理重复的列
# 因为贝叶斯表里有一列叫 'Actual_2021_xxx'，它和 Master 表里的 '2021_xxx_Pct' 是一模一样的
# 为了表格清爽，我们把贝叶斯表里的 Actual 列删掉，保留 Master 表的即可
cols_to_drop = [c for c in df_bayes_pca.columns if c.startswith('Actual_2021_')]
df_bayes_clean = df_bayes_pca.drop(columns=cols_to_drop)

# 4. 世纪大合并 (无缝内连接)
df_universe = pd.merge(df_master, df_bayes_clean, on='Area name', how='inner')

# ==========================================
# 5. 强迫症专属：极其严谨的列排版逻辑
# ==========================================
cols = df_universe.columns.tolist()

# 抽取各类列名
pca_cols = ['2011_PC1', '2011_PC2', '2021_PC1', '2021_PC2']
master_2011_cols = [c for c in cols if '2011' in c and c not in pca_cols]
master_2021_cols = [c for c in cols if '2021' in c and c not in pca_cols]
bayes_cols = [c for c in cols if 'Predicted' in c or 'Uncertainty' in c or 'Error' in c]

# 重新组装顺序：
# [地名] -> [PCA宏观坐标] -> [2011所有绝对人数与比例] -> [2021所有绝对人数与比例] -> [贝叶斯预测与误差]
final_order = ['Area name'] + pca_cols + master_2011_cols + master_2021_cols + bayes_cols
df_universe = df_universe[final_order]

# 6. 导出最终神级大表
file_name_csv = 'UNIVERSE_DATASET_2011_2021.csv'
file_name_excel = 'UNIVERSE_DATASET_2011_2021.xlsx'

df_universe.to_csv(file_name_csv, index=False, encoding='utf-8-sig')

try:
    df_universe.to_excel(file_name_excel, index=False)
    print(f"\n✅ 融合圆满成功！全景数据集已生成 CSV 和 Excel 两个版本！")
    print(f"💾 Excel 版本已保存为: {file_name_excel}")
except:
    print(f"\n✅ 融合圆满成功！全景数据集已保存为: {file_name_csv}")

print(f"📊 这张表现在拥有 {len(df_universe.columns)} 列，囊括了你本次研究的所有心血！")

🌌 正在启动宇宙级大融合：汇集所有绝对人口、百分比、PCA与贝叶斯预测...

✅ 融合圆满成功！全景数据集已生成 CSV 和 Excel 两个版本！
💾 Excel 版本已保存为: UNIVERSE_DATASET_2011_2021.xlsx
📊 这张表现在拥有 49 列，囊括了你本次研究的所有心血！


In [56]:
import pandas as pd
import numpy as np

print("🪄 正在按照顶级学术规范重构并排版【完整 8 大变量】的终极大师表...")

# 1. 读取完整表
df = pd.read_csv('ULTIMATE_MASTER_WITH_POPULATION.csv')
df_perfect = pd.DataFrame()

# ==========================================
# 📍 A. 基础地理信息
# ==========================================
df_perfect['Area_Code'] = df['Area code']
df_perfect['Area_Name'] = df['Area name']

# ==========================================
# 📊 B. 2011 年核心三联组 (8大变量全覆盖)
# ==========================================
# --- 1. 教育 (低、中、高) ---
df_perfect['Total_Edu_2011'] = df['Total_Edu_2011']
df_perfect['EduLow_Abs_2011'] = df['Edu_Low_2011']
df_perfect['Pct_EduLow_2011'] = (df['Edu_Low_2011'] / df['Total_Edu_2011'] * 100).round(1)
df_perfect['EduMedium_Abs_2011'] = df['Edu_Medium_2011']
df_perfect['Pct_EduMedium_2011'] = (df['Edu_Medium_2011'] / df['Total_Edu_2011'] * 100).round(1)
df_perfect['EduHigh_Abs_2011'] = df['Edu_High_2011']
df_perfect['Pct_EduHigh_2011'] = (df['Edu_High_2011'] / df['Total_Edu_2011'] * 100).round(1)

# --- 2. 职业 (优势、中层、劣势) ---
df_perfect['Total_Occ_2011'] = df['Total_Occ_2011']
df_perfect['OccAdvantaged_Abs_2011'] = df['Occ_Advantaged_2011']
df_perfect['Pct_OccAdvantaged_2011'] = (df['Occ_Advantaged_2011'] / df['Total_Occ_2011'] * 100).round(1)
df_perfect['OccIntermediate_Abs_2011'] = df['Occ_Intermediate_2011']
df_perfect['Pct_OccIntermediate_2011'] = (df['Occ_Intermediate_2011'] / df['Total_Occ_2011'] * 100).round(1)
df_perfect['OccDisadvantaged_Abs_2011'] = df['Occ_Disadvantaged_2011']
df_perfect['Pct_OccDisadvantaged_2011'] = (df['Occ_Disadvantaged_2011'] / df['Total_Occ_2011'] * 100).round(1)

# --- 3. 健康 (良好、较差) ---
df_perfect['Total_Health_2011'] = df['Total_Health_2011']
df_perfect['HealthGood_Abs_2011'] = df['Health_Good_2011']
df_perfect['Pct_HealthGood_2011'] = (df['Health_Good_2011'] / df['Total_Health_2011'] * 100).round(1)
df_perfect['HealthPoor_Abs_2011'] = df['Health_Poor_2011']
df_perfect['Pct_HealthPoor_2011'] = (df['Health_Poor_2011'] / df['Total_Health_2011'] * 100).round(1)


# ==========================================
# 📈 C. 2021 年核心三联组 (8大变量全覆盖)
# ==========================================
# --- 1. 教育 ---
df_perfect['Total_Edu_2021'] = df['Total_Edu_2021']
df_perfect['EduLow_Abs_2021'] = df['Edu_Low_2021']
df_perfect['Pct_EduLow_2021'] = (df['Edu_Low_2021'] / df['Total_Edu_2021'] * 100).round(1)
df_perfect['EduMedium_Abs_2021'] = df['Edu_Medium_2021']
df_perfect['Pct_EduMedium_2021'] = (df['Edu_Medium_2021'] / df['Total_Edu_2021'] * 100).round(1)
df_perfect['EduHigh_Abs_2021'] = df['Edu_High_2021']
df_perfect['Pct_EduHigh_2021'] = (df['Edu_High_2021'] / df['Total_Edu_2021'] * 100).round(1)

# --- 2. 职业 ---
df_perfect['Total_Occ_2021'] = df['Total_Occ_2021']
df_perfect['OccAdvantaged_Abs_2021'] = df['Occ_Advantaged_2021']
df_perfect['Pct_OccAdvantaged_2021'] = (df['Occ_Advantaged_2021'] / df['Total_Occ_2021'] * 100).round(1)
df_perfect['OccIntermediate_Abs_2021'] = df['Occ_Intermediate_2021']
df_perfect['Pct_OccIntermediate_2021'] = (df['Occ_Intermediate_2021'] / df['Total_Occ_2021'] * 100).round(1)
df_perfect['OccDisadvantaged_Abs_2021'] = df['Occ_Disadvantaged_2021']
df_perfect['Pct_OccDisadvantaged_2021'] = (df['Occ_Disadvantaged_2021'] / df['Total_Occ_2021'] * 100).round(1)

# --- 3. 健康 ---
df_perfect['Total_Health_2021'] = df['Total_Health_2021']
df_perfect['HealthGood_Abs_2021'] = df['Health_Good_2021']
df_perfect['Pct_HealthGood_2021'] = (df['Health_Good_2021'] / df['Total_Health_2021'] * 100).round(1)
df_perfect['HealthPoor_Abs_2021'] = df['Health_Poor_2021']
df_perfect['Pct_HealthPoor_2021'] = (df['Health_Poor_2021'] / df['Total_Health_2021'] * 100).round(1)


# ==========================================
# 🔮 D. 贝叶斯预测与误差分析 (8大变量全覆盖)
# ==========================================
def add_bayes_columns(prefix, original_name):
    """一个小函数，帮助批量生成预测、误差和区间列"""
    df_perfect[f'Bayes_Pred_{prefix}_2021'] = df[f'{original_name}_Bayes_Pred']
    # 误差 = 2021真实百分比 - 贝叶斯预测百分比
    df_perfect[f'Error_{prefix}_2021'] = (df_perfect[f'Pct_{prefix}_2021'] - df[f'{original_name}_Bayes_Pred']).round(1)
    df_perfect[f'Lower_{prefix}_2021'] = df[f'{original_name}_CI_Lower']
    df_perfect[f'Upper_{prefix}_2021'] = df[f'{original_name}_CI_Upper']

# 教育
add_bayes_columns('EduLow', 'Edu_Low')
add_bayes_columns('EduMedium', 'Edu_Medium')
add_bayes_columns('EduHigh', 'Edu_High')

# 职业
add_bayes_columns('OccAdvantaged', 'Occ_Advantaged')
add_bayes_columns('OccIntermediate', 'Occ_Intermediate')
add_bayes_columns('OccDisadvantaged', 'Occ_Disadvantaged')

# 健康
add_bayes_columns('HealthGood', 'Health_Good')
add_bayes_columns('HealthPoor', 'Health_Poor')


# ==========================================
# 📉 E. 降维坐标系 (PCA)
# ==========================================
df_perfect['PCA_1'] = df['PCA1_2011_Social_Advantage']
df_perfect['PCA_2'] = df['PCA2_2011_Secondary_Axis']


# ==========================================
# 💾 保存神仙排版表格
# ==========================================
output_name = 'FINAL_FORMATTED_TABLE_8VARS_2011_2021.csv'
df_perfect.to_csv(output_name, index=False, encoding='utf-8-sig')

print(f"🎉 8 大变量全量重组成功！真正的最终版！")
print(f"💾 表格已保存为：{output_name}")
print(f"现在的表格就像是一个庞大且精密的数据仪表盘，你可以用它分析任何一个阶层的变化了！")

🪄 正在按照顶级学术规范重构并排版【完整 8 大变量】的终极大师表...
🎉 8 大变量全量重组成功！真正的最终版！
💾 表格已保存为：FINAL_FORMATTED_TABLE_8VARS_2011_2021.csv
现在的表格就像是一个庞大且精密的数据仪表盘，你可以用它分析任何一个阶层的变化了！


In [57]:
import pandas as pd
import numpy as np
from sklearn.linear_model import BayesianRidge
from sklearn.preprocessing import StandardScaler
import umap.umap_ as umap
import warnings
warnings.filterwarnings('ignore')

# 1. 加载你现有的数据表
file_name = 'FINAL_FORMATTED_TABLE_8VARS_2011_2021.csv'
df = pd.read_csv(file_name)

# 定义需要预测的8个核心变量（聚焦于百分比 Pct）
vars_to_predict = [
    'EduLow', 'EduMedium', 'EduHigh',
    'OccAdvantaged', 'OccIntermediate', 'OccDisadvantaged',
    'HealthGood', 'HealthPoor'
]

results = []

print("⏳ 正在运行 2031 年贝叶斯推断 (仅计算预测值)...")
# 2. 遍历每一个地区，基于2011和2021的数据推断2031
for index, row in df.iterrows():
    area_data = {'Area_Code': row['Area_Code'], 'Area_Name': row['Area_Name']}
    
    for var in vars_to_predict:
        # 提取历史数据点
        y_2011 = row[f'Pct_{var}_2011']
        y_2021 = row[f'Pct_{var}_2021']
        
        # 构建时间轴：假设2011为t=0, 2021为t=10, 预测2031即t=20
        X_train = np.array([[0], [10]])
        y_train = np.array([y_2011, y_2021])
        X_test = np.array([[20]])
        
        # 初始化贝叶斯岭回归模型
        model = BayesianRidge()
        model.fit(X_train, y_train)
        
        # 仅预测 2031 的均值 (不再 return_std)
        pred_mean = model.predict(X_test)
        
        # 数值边界截断：百分比不能低于0或高于100
        pred_val = np.clip(pred_mean[0], 0.0, 100.0)
        
        # 保存结果 (只保留预测值)
        area_data[f'Bayes_Pred_{var}_2031'] = pred_val
        
    results.append(area_data)

# 将预测结果转化为 DataFrame
df_2031 = pd.DataFrame(results)

print("🌌 正在进行 2031 年预测数据的 UMAP 降维...")
# 3. 提取刚刚预测好的 2031 核心特征，准备降维
features_2031 = [f'Bayes_Pred_{var}_2031' for var in vars_to_predict]
X_umap = df_2031[features_2031].values

# 标准化数据（降维前必须消除量纲差异）
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_umap)

# 4. 执行 UMAP 降维
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
embedding = reducer.fit_transform(X_scaled)

# 将 UMAP 坐标添加到数据表中
df_2031['UMAP_1'] = embedding[:, 0]
df_2031['UMAP_2'] = embedding[:, 1]

# 5. 合并回原始表格并导出
df_final = pd.merge(df, df_2031, on=['Area_Code', 'Area_Name'])
output_name = 'FINAL_PREDICTIONS_2031_WITH_UMAP_NO_ERROR.csv'
df_final.to_csv(output_name, index=False)

print(f"✅ 大功告成！仅包含 2031 预测值与 UMAP 坐标的新表格已保存至：{output_name}")

E:\Anaconda\Anaconda3\envs\text_analytics\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


⏳ 正在运行 2031 年贝叶斯推断 (仅计算预测值)...
🌌 正在进行 2031 年预测数据的 UMAP 降维...
✅ 大功告成！仅包含 2031 预测值与 UMAP 坐标的新表格已保存至：FINAL_PREDICTIONS_2031_WITH_UMAP_NO_ERROR.csv


In [58]:
import pandas as pd
import numpy as np
from sklearn.linear_model import BayesianRidge
from sklearn.preprocessing import StandardScaler
import umap.umap_ as umap
import warnings
warnings.filterwarnings('ignore')

# 1. 加载现有的完整数据表（里面包含所有的健康数据）
file_name = 'FINAL_FORMATTED_TABLE_8VARS_2011_2021.csv'
df = pd.read_csv(file_name)

# 2. 明确定义：只预测教育和职业这 6 个特征
vars_to_predict = [
    'EduLow', 'EduMedium', 'EduHigh',
    'OccAdvantaged', 'OccIntermediate', 'OccDisadvantaged'
]

results = []

print("⏳ 正在运行 2031 年核心指标预测 (已跳过健康指标)...")
for index, row in df.iterrows():
    # 提取区域主键，用于后续拼接
    area_data = {'Area_Code': row['Area_Code'], 'Area_Name': row['Area_Name']}
    
    for var in vars_to_predict:
        y_2011 = row[f'Pct_{var}_2011']
        y_2021 = row[f'Pct_{var}_2021']
        
        # 贝叶斯推断 2031
        X_train = np.array([[0], [10]])
        y_train = np.array([y_2011, y_2021])
        X_test = np.array([[20]])
        
        model = BayesianRidge()
        model.fit(X_train, y_train)
        
        pred_val = np.clip(model.predict(X_test)[0], 0.0, 100.0)
        area_data[f'Bayes_Pred_{var}_2031'] = pred_val
        
    results.append(area_data)

# 将预测结果转换为独立的数据框
df_2031 = pd.DataFrame(results)

print("🌌 正在进行 2031 核心数据的 UMAP 降维...")
# 提取这 6 个预测特征进行降维
features_2031 = [f'Bayes_Pred_{var}_2031' for var in vars_to_predict]
X_umap = df_2031[features_2031].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_umap)

reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
embedding = reducer.fit_transform(X_scaled)

# 将坐标加入 2031 数据框
df_2031['UMAP_1'] = embedding[:, 0]
df_2031['UMAP_2'] = embedding[:, 1]

print("🔗 正在拼接历史原表数据...")
# 3. 终极拼接：把原始大表 (含所有健康数据) 和刚做好的 2031 表合并
df_final = pd.merge(df, df_2031, on=['Area_Code', 'Area_Name'], how='left')

output_name = 'FINAL_DATA_WITH_HEALTH_AND_2031_UMAP.csv'
df_final.to_csv(output_name, index=False)

print(f"✅ 大功告成！\n原始健康数据已保留，且仅添加了教育/职业的 2031 预测及 UMAP。\n文件保存在：{output_name}")

⏳ 正在运行 2031 年核心指标预测 (已跳过健康指标)...
🌌 正在进行 2031 核心数据的 UMAP 降维...
🔗 正在拼接历史原表数据...
✅ 大功告成！
原始健康数据已保留，且仅添加了教育/职业的 2031 预测及 UMAP。
文件保存在：FINAL_DATA_WITH_HEALTH_AND_2031_UMAP.csv


In [60]:
import pandas as pd
from sklearn.cluster import KMeans

# 1. 读取你刚才跑完的那张终极数据表
file_name = 'FINAL_DATA_WITH_HEALTH_AND_2031_UMAP.csv'
df = pd.read_csv(file_name)

print("正在对 UMAP 坐标进行 KMeans 聚类...")
# 2. 针对 UMAP_1 和 UMAP_2 进行聚类
# n_clusters=2 表示分为两类，完美对应你参考图里的 蓝色(Cluster 1) 和 橙色(Cluster 2)
kmeans = KMeans(n_clusters=2, random_state=42)
df['Cluster'] = kmeans.fit_predict(df[['UMAP_1', 'UMAP_2']])

# 3. 将机器算出来的数字 0 和 1，映射成高大上的文字标签
df['Cluster'] = df['Cluster'].map({0: 'Cluster 1', 1: 'Cluster 2'})

# 4. 保存为带有聚类标签的最终版本
output_name = 'FINAL_DATA_WITH_CLUSTER.csv'
df.to_csv(output_name, index=False)

print(f"✅ 聚类完成！新增的 'Cluster' 变量已保存至：{output_name}")

正在对 UMAP 坐标进行 KMeans 聚类...
✅ 聚类完成！新增的 'Cluster' 变量已保存至：FINAL_DATA_WITH_CLUSTER.csv


In [4]:
import pandas as pd
from sklearn.linear_model import BayesianRidge

# 1. 读取你当前的数据表
df = pd.read_csv('FINAL_DATA_WITH_CLUSTER.csv')

# 2. 设定自变量(X)和因变量(Y)，全部使用 2021 年真实数据
X = df[['Pct_EduHigh_2021', 'Pct_HealthGood_2021']]
y = df['Pct_OccAdvantaged_2021']

# 3. 建立并训练贝叶斯回归模型
bayes_model = BayesianRidge()
bayes_model.fit(X, y)

# 4. 查看模型学到的系数 (谁的影响力更大？)
print(f"高等教育的权重 (Beta 1): {bayes_model.coef_[0]:.4f}")
print(f"良好健康的权重 (Beta 2): {bayes_model.coef_[1]:.4f}")

# 5. 生成预测值和误差，并作为新列加入表格
df['Bayes_Struct_Pred_Occ_2021'] = bayes_model.predict(X)
df['Struct_Error_Occ_2021'] = df['Pct_OccAdvantaged_2021'] - df['Bayes_Struct_Pred_Occ_2021']

# 6. 保存为新的终极表格
df.to_csv('FINAL_DATA_WITH_STRUCT_BAYES.csv', index=False)
print("贝叶斯结构预测完成，已保存至 FINAL_DATA_WITH_STRUCT_BAYES.csv！")

高等教育的权重 (Beta 1): 0.6023
良好健康的权重 (Beta 2): 0.1826
贝叶斯结构预测完成，已保存至 FINAL_DATA_WITH_STRUCT_BAYES.csv！
